In [7]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [8]:
import random
import time
import joblib
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pandas as pd

from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import KerasClassifier


In [9]:
# Configuración de visualización

sns.set_style("whitegrid")

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
MLP_ATTACK_DEVICE = TRAIN_DEVICE
CNN_ATTACK_DEVICE = "/CPU:0"
INFER_DEVICE = "/CPU:0"
ATTACK_BATCH_SIZE = 128

SEED = 42
MODEL_DIR = os.path.join("model", "ton-iot")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

if HAS_GPU:
    print("GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.")
else:
    print("No hay GPU disponible. Todo el notebook se ejecutará en CPU.")

tf.keras.backend.clear_session()

def build_mlp_model(input_dim, hidden_units):
    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape=(input_dim,)))
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model


GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.


In [10]:
# ==========================================
# 1. CARGA DE DATOS
# ==========================================

TARGET_COL = "label"
path_df = "../../DATASETS/dataSets_Reducidos/ton_iot/datos_TON_IoT_redux.csv"

df = pl.read_csv(path_df).drop_nulls()

cols_to_drop = [
    "label",
    "type",
    "src_ip",
    "dst_ip"
]

X = df.drop(cols_to_drop).to_pandas()
y_np = df[TARGET_COL].to_numpy().astype(np.int8)

indices = np.arange(len(X))

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np[train_full_idx],
)

X_train_full_df = X.iloc[train_full_idx].copy()
X_test_df = X.iloc[test_idx].copy()
X_train_df = X.iloc[train_idx].copy()
X_val_df = X.iloc[val_idx].copy()

y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

categorical_cols = ["proto", "conn_state"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

X_train_np = preprocessor.fit_transform(X_train_df).astype(np.float32)
X_val_np = preprocessor.transform(X_val_df).astype(np.float32)
X_test_np = preprocessor.transform(X_test_df).astype(np.float32)
X_full_train_np = preprocessor.transform(X_train_full_df).astype(np.float32)

feature_columns = preprocessor.get_feature_names_out().tolist()

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X):,}")
print("La clase 0 corresponde a Benign y la clase 1 a Malicious.")
print("Shape transformado:", X_full_train_np.shape)


Entrenamiento: 135,067 muestras
Validación:    33,767 muestras
Test:          42,209 muestras
Clases en test: [0 1]
Total muestras: 211,043
La clase 0 corresponde a Benign y la clase 1 a Malicious.
Shape transformado: (168834, 23)


In [11]:
# ==========================================
# 2. CONFIGURACIÓN DE LOS GANADORES
# ==========================================

RF_CONFIG = {"n_estimators": 50, "max_depth": 18}
XGB_CONFIG = {"n_estimators": 50, "max_depth": 7, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 100, "num_leaves": 233, "max_depth": 11, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 250, "depth": 9, "learning_rate": 0.1}

SVM_C = 2.870875
MLP_INPUT_DIM = X_full_train_np.shape[1]
MLP_HIDDEN_UNITS = (64, 16, 64)
CNN1D_CONFIG = {"nf": 96, "k": 5, "d": 96}

In [12]:
# ==========================================
# 3. CARGA DE MODELOS ENTRENADOS
# ==========================================

MODEL_PATHS = {
    "rf": os.path.join(MODEL_DIR, "rf_ton_iot.joblib"),
    "xgb": os.path.join(MODEL_DIR, "xgb_ton_iot.joblib"),
    "lgbm": os.path.join(MODEL_DIR, "lgbm_ton_iot.joblib"),
    "catboost": os.path.join(MODEL_DIR, "catboost_ton_iot.joblib"),
    "svm": os.path.join(MODEL_DIR, "svm_ton_iot.joblib"),
    "mlp": os.path.join(MODEL_DIR, "mlp_ton_iot.joblib"),
    "cnn": os.path.join(MODEL_DIR, "cnn_ton_iot.joblib"),
}

def load_model(model_key):
    model_path = MODEL_PATHS[model_key]
    print(f"Cargando {model_key} desde {model_path}...")
    return joblib.load(model_path)

rf_model = load_model("rf")
xgb_model = load_model("xgb")
lgbm_model = load_model("lgbm")
cat_model = load_model("catboost")


Cargando rf desde model/ton-iot/rf_ton_iot.joblib...
Cargando xgb desde model/ton-iot/xgb_ton_iot.joblib...
Cargando lgbm desde model/ton-iot/lgbm_ton_iot.joblib...
Cargando catboost desde model/ton-iot/catboost_ton_iot.joblib...


In [13]:
# ==========================================
# 4. PREPARACIÓN DE VISTAS Y MODELOS DIFERENCIABLES
# ==========================================

svm_model = load_model("svm")

print("\nPreparando vistas del dataset para cada familia de modelos...")
X_test_np_arr = np.array(X_test_np, dtype=np.float32)

mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train_np).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np_arr).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train_np).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np_arr).astype(np.float32)
X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)

DATASET_VIEWS = {
    "raw": {"train": X_full_train_np, "test": X_test_np_arr},
    "standard": {"train": X_train_scaled_mlp, "test": X_test_scaled_mlp},
    "minmax": {"train": X_train_scaled_cnn, "test": X_test_scaled_cnn},
}

print("Vistas disponibles:", ", ".join(DATASET_VIEWS.keys()))

with tf.device(MLP_ATTACK_DEVICE):
    mlp_model = load_model("mlp")

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM, MLP_HIDDEN_UNITS)

def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)

def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()

with tf.device(CNN_ATTACK_DEVICE):
    cnn_model = load_model("cnn")

cnn_model_cpu = clone_keras_model_to_cpu(
    build_cnn1d_model,
    cnn_model,
    X_train_tabular_cnn.shape[1],
    CNN1D_CONFIG["nf"],
    CNN1D_CONFIG["k"],
    CNN1D_CONFIG["d"]
)

def cnn_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = cnn_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)


Cargando svm desde model/ton-iot/svm_ton_iot.joblib...

Preparando vistas del dataset para cada familia de modelos...
Vistas disponibles: raw, standard, minmax
Cargando mlp desde model/ton-iot/mlp_ton_iot.joblib...


I0000 00:00:1779220064.672386 1459274 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 42720 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:e1:00.0, compute capability: 8.9


Cargando cnn desde model/ton-iot/cnn_ton_iot.joblib...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Una vez que los modelos ya están entrenados, entramos en la fase de evaluación adversaria y transferencia entre espacios `raw`, `standard` y `minmax`.


In [14]:
# Extraemos restricciones tabulares a partir del train transformado
feature_names = feature_columns

X_full_train_df = pl.DataFrame(X_full_train_np, schema=feature_names)

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": X_full_train_df.min().row(0),
    "max": X_full_train_df.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

clip_values_raw = (feature_mins, feature_maxs)

feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)

feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).reshape(-1, 1).astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).reshape(-1, 1).astype(np.float32)

one_hot_columns = [
    col for col in feature_names
    if ("proto_" in col) or ("conn_state_" in col)
]

attack_mask = np.array(
    [0.0 if col in one_hot_columns else 1.0 for col in feature_names],
    dtype=np.float32
)
attack_mask_cnn = attack_mask.reshape(-1, 1).astype(np.float32)

tiny = np.float32(1e-6)
feature_maxs_mlp = np.where(feature_maxs_mlp <= feature_mins_mlp, feature_mins_mlp + tiny, feature_maxs_mlp)
feature_maxs_cnn = np.where(feature_maxs_cnn <= feature_mins_cnn, feature_mins_cnn + tiny, feature_maxs_cnn)

clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para TON-IoT:")
display(tabular_constraints_df)

print("Columnas one-hot bloqueadas:")
print(one_hot_columns)


Restricciones tabulares extraidas para TON-IoT:


feature,min,max
str,f64,f64
"""num__src_port""",-2.001115,1.393837
"""num__dst_port""",-0.343629,6.068274
"""num__duration""",-0.013653,157.152008
"""num__src_bytes""",-0.015531,206.705887
"""num__dst_bytes""",-0.016619,235.662842
…,…,…
"""cat__conn_state_S2""",0.0,1.0
"""cat__conn_state_S3""",0.0,1.0
"""cat__conn_state_SF""",0.0,1.0


Columnas one-hot bloqueadas:
['cat__proto_icmp', 'cat__proto_tcp', 'cat__proto_udp', 'cat__conn_state_OTH', 'cat__conn_state_REJ', 'cat__conn_state_RSTO', 'cat__conn_state_RSTOS0', 'cat__conn_state_RSTR', 'cat__conn_state_RSTRH', 'cat__conn_state_S0', 'cat__conn_state_S1', 'cat__conn_state_S2', 'cat__conn_state_S3', 'cat__conn_state_SF', 'cat__conn_state_SH', 'cat__conn_state_SHR']


In [15]:
# ==========================================
# FASE 1. ENVOLVER LOS MODELOS (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model,
    clip_values=clip_values_mlp,
    use_logits=False
)

print("Envolviendo el modelo CNN en ART con restricciones tabulares...")

clasificador_art_cnn = KerasClassifier(
    model=cnn_model,
    clip_values=clip_values_cnn,
    use_logits=False
)


Envolviendo el modelo MLP en ART con restricciones tabulares...
Envolviendo el modelo CNN en ART con restricciones tabulares...


In [16]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

EPS_VALUES = [0.01, 0.03, 0.05, 0.075, 0.1, 0.2, 0.5]

MODELOS_TABLA = ["RF", "XGB", "LGBM", "CatBoost", "SVM", "MLP", "CNN"]

modelos_clasicos = {
    "RF": rf_model,
    "XGB": xgb_model,
    "LGBM": lgbm_model,
    "CatBoost": cat_model,
    "SVM": svm_model,
}

tiny_step = np.float32(1e-6)


In [17]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_fgsm_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando FGSM sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_vector_mlp = eps_vector_mlp * attack_mask
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, tiny_step).astype(np.float32)

    ataque_fgsm_mlp = FastGradientMethod(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        batch_size=ATTACK_BATCH_SIZE,
    )

    with tf.device(MLP_ATTACK_DEVICE):
        x_test_fgsm_mlp = ataque_fgsm_mlp.generate(
            x=x_test_mlp_attack,
            mask=attack_mask
        )

    x_test_fgsm_mlp_std = x_test_fgsm_mlp.astype(np.float32)
    x_test_fgsm_mlp_raw = mlp_scaler.inverse_transform(x_test_fgsm_mlp_std).astype(np.float32)
    x_test_fgsm_mlp_minmax = cnn_scaler.transform(x_test_fgsm_mlp_raw).astype(np.float32)
    x_test_fgsm_mlp_cnn = x_test_fgsm_mlp_minmax.reshape(
        x_test_fgsm_mlp_minmax.shape[0],
        x_test_fgsm_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_mlp_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_fgsm_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_fgsm_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_fgsm_mlp.append(fila_resultados)

print("Evaluacion FGSM asociada al MLP completada.")


Generando FGSM sobre MLP para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [21:47:49] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
I0000 00:00:1779220069.350025 1460200 service.cc:153] XLA service 0x7f8ea4635960 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779220069.350064 14

Evaluacion FGSM asociada al MLP completada.


In [18]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_fgsm_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando FGSM sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_vector_cnn = eps_vector_cnn * attack_mask_cnn
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, tiny_step).astype(np.float32)

    ataque_fgsm_cnn = FastGradientMethod(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        batch_size=ATTACK_BATCH_SIZE,
    )

    with tf.device(CNN_ATTACK_DEVICE):
        x_test_fgsm_cnn = ataque_fgsm_cnn.generate(
            x=x_test_cnn_attack,
            mask=attack_mask_cnn
        )

    x_test_fgsm_cnn_cnn = x_test_fgsm_cnn.astype(np.float32)
    x_test_fgsm_cnn_minmax = x_test_fgsm_cnn_cnn.reshape(
        x_test_fgsm_cnn_cnn.shape[0],
        x_test_fgsm_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_fgsm_cnn_raw = cnn_scaler.inverse_transform(x_test_fgsm_cnn_minmax).astype(np.float32)
    x_test_fgsm_cnn_std = mlp_scaler.transform(x_test_fgsm_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_fgsm_cnn_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_fgsm_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_fgsm_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_fgsm_cnn.append(fila_resultados)

print("Evaluacion FGSM asociada a la CNN completada.")


Generando FGSM sobre CNN para varios limites de perturbacion...


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have va

Evaluacion FGSM asociada a la CNN completada.


In [19]:
import pandas as pd

tabla_f1_fgsm_mlp = pd.DataFrame(resultados_fgsm_mlp).set_index("Limite perturbacion")
tabla_f1_fgsm_mlp = tabla_f1_fgsm_mlp[MODELOS_TABLA].round(4)

tabla_f1_fgsm_cnn = pd.DataFrame(resultados_fgsm_cnn).set_index("Limite perturbacion")
tabla_f1_fgsm_cnn = tabla_f1_fgsm_cnn[MODELOS_TABLA].round(4)

print("Tabla FGSM asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_fgsm_mlp)

print("Tabla FGSM asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_fgsm_cnn)


Tabla FGSM asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.6799,0.5647,0.6577,0.3210,0.4029,0.3369,0.8454
0.030,0.6425,0.5620,0.6391,0.4003,0.3612,0.2857,0.6518
0.050,0.6690,0.5517,0.5496,0.4257,0.2852,0.2861,0.6237
0.075,0.6807,0.5510,0.6003,0.4358,0.2329,0.2867,0.6239
0.100,0.6630,0.5453,0.6004,0.4570,0.2034,0.3372,0.5768
0.200,0.5255,0.5568,0.5666,0.3917,0.2041,0.3531,0.7194
0.500,0.5082,0.5327,0.6206,0.3025,0.2044,0.2107,0.6785


Tabla FGSM asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.5548,0.6436,0.8253,0.3025,0.3217,0.4965,0.7411
0.030,0.4435,0.6053,0.7713,0.3184,0.3209,0.4768,0.6186
0.050,0.4459,0.6052,0.7218,0.3238,0.3066,0.4412,0.5818
0.075,0.4469,0.6118,0.7506,0.3308,0.2954,0.4406,0.5817
0.100,0.4510,0.6099,0.7504,0.3363,0.2888,0.4637,0.5781
0.200,0.4571,0.6002,0.6939,0.3458,0.2204,0.4591,0.5447
0.500,0.4643,0.6905,0.6761,0.3386,0.2178,0.3854,0.6591


In [20]:
# ========================================================
# FASE 5. LANZAR ATAQUE PGD PARA VARIOS LIMITES DE PERTURBACION
# ========================================================

PGD_MAX_ITER = 20

print("Configurando PGD para varios limites de perturbacion...")
print(f"Iteraciones PGD: {PGD_MAX_ITER}")


Configurando PGD para varios limites de perturbacion...
Iteraciones PGD: 20


In [21]:
feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]

resultados_pgd_mlp = []
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

print("Generando PGD sobre MLP para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_mlp = (eps_base * feature_ranges_mlp).astype(np.float32)
    eps_vector_mlp = eps_vector_mlp * attack_mask
    eps_step_vector_mlp = np.where(eps_vector_mlp > 0, eps_vector_mlp / 4.0, tiny_step).astype(np.float32)

    ataque_pgd_mlp = ProjectedGradientDescent(
        estimator=clasificador_art_mlp,
        eps=eps_vector_mlp,
        eps_step=eps_step_vector_mlp,
        max_iter=PGD_MAX_ITER,
        batch_size=ATTACK_BATCH_SIZE,
    )

    with tf.device(MLP_ATTACK_DEVICE):
        x_test_pgd_mlp = ataque_pgd_mlp.generate(
            x=x_test_mlp_attack,
            mask=attack_mask
        )

    x_test_pgd_mlp_std = x_test_pgd_mlp.astype(np.float32)
    x_test_pgd_mlp_raw = mlp_scaler.inverse_transform(x_test_pgd_mlp_std).astype(np.float32)
    x_test_pgd_mlp_minmax = cnn_scaler.transform(x_test_pgd_mlp_raw).astype(np.float32)
    x_test_pgd_mlp_cnn = x_test_pgd_mlp_minmax.reshape(
        x_test_pgd_mlp_minmax.shape[0],
        x_test_pgd_mlp_minmax.shape[1],
        1
    )

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_mlp_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_mlp_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_mlp_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_pgd_mlp.append(fila_resultados)

print("Evaluacion PGD asociada al MLP completada.")


Generando PGD sobre MLP para varios limites de perturbacion...


PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.60it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.74it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.83it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.71it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/pyt

Evaluacion PGD asociada al MLP completada.


In [22]:
feature_ranges_cnn = clip_values_cnn[1] - clip_values_cnn[0]

resultados_pgd_cnn = []
x_test_cnn_attack = X_test_tabular_cnn.astype(np.float32)

print("Generando PGD sobre CNN para varios limites de perturbacion...")

for eps_base in EPS_VALUES:
    eps_vector_cnn = (eps_base * feature_ranges_cnn).astype(np.float32)
    eps_vector_cnn = eps_vector_cnn * attack_mask_cnn
    eps_step_vector_cnn = np.where(eps_vector_cnn > 0, eps_vector_cnn / 4.0, tiny_step).astype(np.float32)

    ataque_pgd_cnn = ProjectedGradientDescent(
        estimator=clasificador_art_cnn,
        eps=eps_vector_cnn,
        eps_step=eps_step_vector_cnn,
        max_iter=PGD_MAX_ITER,
        batch_size=ATTACK_BATCH_SIZE,
    )

    with tf.device(CNN_ATTACK_DEVICE):
        x_test_pgd_cnn = ataque_pgd_cnn.generate(
            x=x_test_cnn_attack,
            mask=attack_mask_cnn
        )

    x_test_pgd_cnn_cnn = x_test_pgd_cnn.astype(np.float32)
    x_test_pgd_cnn_minmax = x_test_pgd_cnn_cnn.reshape(
        x_test_pgd_cnn_cnn.shape[0],
        x_test_pgd_cnn_cnn.shape[1]
    ).astype(np.float32)
    x_test_pgd_cnn_raw = cnn_scaler.inverse_transform(x_test_pgd_cnn_minmax).astype(np.float32)
    x_test_pgd_cnn_std = mlp_scaler.transform(x_test_pgd_cnn_raw).astype(np.float32)

    fila_resultados = {"Limite perturbacion": eps_base}

    for nombre_modelo in MODELOS_TABLA:
        if nombre_modelo in ["RF", "XGB", "LGBM", "CatBoost", "SVM"]:
            y_pred = modelos_clasicos[nombre_modelo].predict(x_test_pgd_cnn_raw)
        elif nombre_modelo == "MLP":
            y_pred = mlp_predict_labels(x_test_pgd_cnn_std)
        elif nombre_modelo == "CNN":
            y_pred = cnn_predict_labels(x_test_pgd_cnn_cnn)

        fila_resultados[nombre_modelo] = f1_score(y_test_np, y_pred, pos_label=0)

    resultados_pgd_cnn.append(fila_resultados)

print("Evaluacion PGD asociada a la CNN completada.")


Generando PGD sobre CNN para varios limites de perturbacion...


PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  3.38it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/pyt

Evaluacion PGD asociada a la CNN completada.


In [23]:
tabla_f1_pgd_mlp = pd.DataFrame(resultados_pgd_mlp).set_index("Limite perturbacion")
tabla_f1_pgd_mlp = tabla_f1_pgd_mlp[MODELOS_TABLA].round(4)

tabla_f1_pgd_cnn = pd.DataFrame(resultados_pgd_cnn).set_index("Limite perturbacion")
tabla_f1_pgd_cnn = tabla_f1_pgd_cnn[MODELOS_TABLA].round(4)

print("Tabla PGD asociada a la perturbacion generada sobre el MLP")
display(tabla_f1_pgd_mlp)

print("Tabla PGD asociada a la perturbacion generada sobre la CNN")
display(tabla_f1_pgd_cnn)


Tabla PGD asociada a la perturbacion generada sobre el MLP


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.7322,0.4618,0.6258,0.4997,0.7548,0.3946,0.9192
0.030,0.5675,0.4086,0.5527,0.2887,0.7922,0.4104,0.6605
0.050,0.5725,0.3994,0.5433,0.3501,0.7608,0.3850,0.6488
0.075,0.5221,0.3537,0.6107,0.3290,0.6546,0.3831,0.6317
0.100,0.4931,0.3531,0.5877,0.3367,0.5737,0.3830,0.6114
0.200,0.4590,0.3066,0.5526,0.2307,0.4415,0.3826,0.4588
0.500,0.3671,0.3278,0.4360,0.1261,0.3610,0.3826,0.0817


Tabla PGD asociada a la perturbacion generada sobre la CNN


,RF,XGB,LGBM,CatBoost,SVM,MLP,CNN
Limite perturbacion,,,,,,,
0.010,0.5529,0.5838,0.7368,0.3758,0.3273,0.5400,0.7410
0.030,0.5078,0.6789,0.7228,0.3322,0.3208,0.4126,0.6080
0.050,0.5977,0.4666,0.6414,0.4006,0.4115,0.4366,0.5804
0.075,0.4611,0.4123,0.5110,0.3748,0.4057,0.4201,0.5501
0.100,0.5492,0.4199,0.4765,0.4390,0.4901,0.4025,0.5360
0.200,0.4576,0.4317,0.4569,0.3485,0.4224,0.3734,0.5026
0.500,0.4910,0.3701,0.4295,0.4782,0.6064,0.3919,0.4413
